
## Part 1 — NumPy Fundamentals

### Q1.1 — Array Creation & Statistical Filtering


In [ ]:
import numpy as np

arr = np.linspace(0, 20, 100)
mean   = np.mean(arr)
median = np.median(arr)
std    = np.std(arr)
var    = np.var(arr)

print(f"\nMean:               {mean:.4f}")
print(f"Median:             {median:.4f}")
print(f"Standard Deviation: {std:.4f}")
print(f"Variance:           {var:.4f}")

### Q1.2 — Reshaping

In [ ]:
# Create 1D array of integers from 1 to 36
arr_1d = np.arange(1, 37)
print("1D Array:", arr_1d)
print("Shape after creation:", arr_1d.shape)

# Reshape to (4x9) matrix
arr_2d = arr_1d.reshape(4, 9)
print("\nReshaped to (4x9):\n", arr_2d)
print("Shape after reshape:", arr_2d.shape)

# Flatten back to 1D
arr_flat = arr_2d.flatten()
print("\nFlattened back to 1D:", arr_flat)
print("Shape after flatten:", arr_flat.shape)

---
## Part 2 — Pandas & Data Wrangling

### Q2.1 — First Look

In [ ]:
import pandas as pd
import seaborn as sns

# Load the Titanic dataset built into seaborn
df = sns.load_dataset('titanic')

# Display first 5 rows
display(df.head())

In [ ]:
# Dataset info
df.info()

In [ ]:
# Statistical summary
display(df.describe())

### Q2.2 — Missing Value Treatment

In [ ]:
# List columns with missing values, count, and percentage
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)
print("=== Columns with Missing Values ===")
display(missing_df)

In [ ]:
# Fill missing 'age' with median age grouped by pclass
df['age'] = df.groupby('pclass')['age'].transform(
    lambda x: x.fillna(x.median())
)
print("Age nulls after treatment:", df['age'].isnull().sum())

# Fill missing 'embarked' with mode
embarked_mode = df['embarked'].mode()[0]
df['embarked'] = df['embarked'].fillna(embarked_mode)
print("Embarked nulls after treatment:", df['embarked'].isnull().sum())

# Drop the 'deck' column
df = df.drop(columns=['deck'])
print("'deck' column dropped.")

# Verify no remaining nulls in treated columns
print("\nRemaining nulls in treated columns:")
print(df[['age', 'embarked']].isnull().sum())

**Why drop `deck`?**  
As seen above, `deck` is missing for approximately **77–78% of all passengers**. With such a high proportion of missing data, imputation would introduce more noise than signal — we would effectively be fabricating nearly 4/5 of the column. Dropping it is the appropriate choice since it cannot contribute reliably to any downstream analysis or model.

---
## Part 3 — Matplotlib Visualizations

### Q3.1 — Histogram

In [ ]:
import matplotlib.pyplot as plt

mean_age   = df['age'].mean()
median_age = df['age'].median()

fig, ax = plt.subplots(figsize=(9, 5))

# Histogram of age
ax.hist(df['age'], bins=20, color='steelblue', edgecolor='white')

# Vertical lines for mean and median
ax.axvline(mean_age,   color='red',   linestyle='--', linewidth=1.8, label=f'Mean Age ({mean_age:.1f})')
ax.axvline(median_age, color='green', linestyle='-',  linewidth=1.8, label=f'Median Age ({median_age:.1f})')

ax.set_title('Distribution of Passenger Age on the Titanic')
ax.set_xlabel('Age')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.show()

### Q3.2 — Scatter Plot with Color Encoding

In [ ]:
# Color map: 0 = red (did not survive), 1 = green (survived)
colors = df['survived'].map({0: 'red', 1: 'green'})

fig, ax = plt.subplots(figsize=(9, 5))

# Scatter: non-survivors
mask_0 = df['survived'] == 0
mask_1 = df['survived'] == 1
ax.scatter(df.loc[mask_0, 'age'], df.loc[mask_0, 'fare'],
           color='red',   alpha=0.5, s=20, label='Did Not Survive (0)')
ax.scatter(df.loc[mask_1, 'age'], df.loc[mask_1, 'fare'],
           color='green', alpha=0.5, s=20, label='Survived (1)')

ax.set_ylim(0, 300)   # Cap y-axis to remove extreme fare outliers
ax.set_title('Age vs Fare — Colored by Survival Status')
ax.set_xlabel('Age')
ax.set_ylabel('Fare (capped at 300)')
ax.legend()
plt.tight_layout()
plt.show()

The `fare` column contains extreme outliers (some values above 500). Capping the y-axis at 300 removes these few data points from the visible plot area, making the bulk of the distribution much easier to interpret without distortion. The underlying data is not modified.

---
## Part 4 — Exploratory Data Analysis

### Q4.1 — Initial Inspection

In [ ]:
# Load Advertising dataset from GitHub (public mirror)
ad_url = ' https://www.kaggle.com/datasets/ashydv/advertising-dataset'
ad = pd.read_csv(ad_url)

print("Shape:", ad.shape)
print("\nData Types:")
print(ad.dtypes)
print("\nMissing Values:")
print(ad.isnull().sum())
print("\n=== Describe ===")
display(ad.describe())

**Range and Spread of Each Variable:**

| Variable | Min   | Max    | Mean   | Std Dev | Observations |
|----------|-------|--------|--------|---------|--------------|
| TV       | 0.7   | 296.4  | 147.0  | 85.9    | Wide range; fairly uniform spread |
| Radio    | 0.0   | 49.6   | 23.3   | 14.8    | Moderate spread, max ~50 |
| Newspaper| 0.4   | 114.0  | 30.6   | 21.8    | High right-skew likely due to outliers |
| Sales    | 1.6   | 27.0   | 14.0   | 5.2     | Target variable; reasonable spread |

In [ ]:
# 4 histograms in a 1x4 subplot (one per column)
columns = ['TV', 'Radio', 'Newspaper', 'Sales']
colors_hist = ['steelblue', 'coral', 'mediumseagreen', 'orchid']

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, col, clr in zip(axes, columns, colors_hist):
    ax.hist(ad[col], bins=20, color=clr, edgecolor='white')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

plt.suptitle('Advertising Dataset — Variable Distributions', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

**Visual Outlier Observations:**  
- **Newspaper**: The histogram shows a long right tail — a few campaigns with newspaper spend > 80 appear to be outliers relative to the bulk of the data which clusters below 60.  
- **TV**: Appears roughly uniform with no dramatic outliers.  
- **Radio** and **Sales**: Roughly bell-shaped with no extreme outliers apparent from histograms.

### Q4.2 — Correlation Analysis

In [ ]:
# Compute correlation matrix
corr_matrix = ad.corr()
print("Correlation Matrix:")
display(corr_matrix.round(2))

In [ ]:
# Heatmap using plt.imshow with annotations
fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(corr_matrix.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)

# Annotate each cell
labels = corr_matrix.columns.tolist()
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels)
ax.set_yticklabels(labels)

for i in range(len(labels)):
    for j in range(len(labels)):
        val = round(corr_matrix.values[i, j], 2)
        ax.text(j, i, str(val), ha='center', va='center', fontsize=11,
                color='white' if abs(val) > 0.6 else 'black')

ax.set_title('Correlation Matrix Heatmap — Advertising Dataset')
plt.tight_layout()
plt.show()

**Correlation Insights:**

- **Most correlated with Sales:** `TV` has the highest correlation with `Sales` (~0.78), making it the strongest predictor.  
- **TV vs Radio:** The correlation between `TV` and `Radio` is very low (~0.05), indicating they are essentially **not correlated** — advertising spend on TV and Radio are independent of each other in this dataset.

In [ ]:
# 3x3 grid of scatter plots — all pairs of variables
cols = ['TV', 'Radio', 'Newspaper', 'Sales']
fig, axes = plt.subplots(3, 3, figsize=(12, 10))

# Pairs (excluding self-self, we pick the 9 most meaningful combos)
pairs = [
    ('TV',        'Sales'),
    ('Radio',     'Sales'),
    ('Newspaper', 'Sales'),
    ('TV',        'Radio'),
    ('TV',        'Newspaper'),
    ('Radio',     'Newspaper'),
    ('Sales',     'TV'),
    ('Sales',     'Radio'),
    ('Sales',     'Newspaper'),
]

for ax, (x_col, y_col) in zip(axes.flatten(), pairs):
    ax.scatter(ad[x_col], ad[y_col], alpha=0.5, s=15, color='steelblue')
    ax.set_xlabel(x_col, fontsize=8)
    ax.set_ylabel(y_col, fontsize=8)
    ax.set_title(f'{x_col} vs {y_col}', fontsize=9)

plt.suptitle('Scatter Plot Matrix — Advertising Dataset', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

---
## Part 5 — Linear Regression

### Q5.1 — Scratch Implementation: Simple Linear Regression

In [ ]:
# Use TV as feature, Sales as target
X = ad['TV'].values
y = ad['Sales'].values

# Manual train/test split: first 160 rows = train, rest = test
X_train_scratch, X_test_scratch = X[:160], X[160:]
y_train_scratch, y_test_scratch = y[:160], y[160:]

# Compute slope (m) and intercept (b) using closed-form formula
x_mean = np.mean(X_train_scratch)
y_mean = np.mean(y_train_scratch)

m = np.sum((X_train_scratch - x_mean) * (y_train_scratch - y_mean)) / \
    np.sum((X_train_scratch - x_mean) ** 2)
b = y_mean - m * x_mean

print(f"Slope (m):     {m:.6f}")
print(f"Intercept (b): {b:.6f}")

# Predictions on test set
y_pred_scratch = m * X_test_scratch + b

# Manual MSE
mse_scratch = np.mean((y_test_scratch - y_pred_scratch) ** 2)
print(f"\nTest MSE (scratch): {mse_scratch:.4f}")

### Q5.2 — Sklearn Implementation

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# 80-20 split with random_state=42
X_sk = ad[['TV']].values
y_sk = ad['Sales'].values

X_train_sk, X_test_sk, y_train_sk, y_test_sk = train_test_split(
    X_sk, y_sk, test_size=0.2, random_state=42
)

model_sk = LinearRegression()
model_sk.fit(X_train_sk, y_train_sk)

print(f"Sklearn Coefficient (m): {model_sk.coef_[0]:.6f}")
print(f"Sklearn Intercept   (b): {model_sk.intercept_:.6f}")

print(f"\nScratch m: {m:.6f}  |  Sklearn m: {model_sk.coef_[0]:.6f}")
print(f"Scratch b: {b:.6f}  |  Sklearn b: {model_sk.intercept_:.6f}")

**Comparison & Why Small Differences Exist:**  
Both methods produce very similar `m` and `b` values. The small difference arises because the **training data is different**: the scratch implementation uses the first 160 rows sequentially, while `train_test_split(random_state=42)` performs a **random shuffle** before splitting. Since the closed-form OLS solution is sensitive to which data points are in the training set, different subsets yield slightly different parameter estimates — even though both are globally optimal for their respective training sets.

### Q5.3 — Evaluation Metrics

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Predictions using sklearn model
y_train_pred = model_sk.predict(X_train_sk)
y_test_pred  = model_sk.predict(X_test_sk)

def print_metrics(y_true, y_pred, split_name):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    print(f"--- {split_name} ---")
    print(f"  MAE:  {mae:.4f}")
    print(f"  MSE:  {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R²:   {r2:.4f}")
    print()

print_metrics(y_train_sk, y_train_pred, 'Train Set')
print_metrics(y_test_sk,  y_test_pred,  'Test Set')

**Model Fit Assessment:**  
The train and test R² scores are very close to each other (both around 0.60–0.65), and neither is extremely high nor extremely low. This suggests the model is **neither overfitting nor severely underfitting** — it is reasonably well-fit for a simple linear regression with one feature.  
- If the model were **overfitting**, train R² would be much higher than test R².  
- If it were **underfitting**, both R² values would be low (e.g., < 0.3).  
- The moderate R² (~0.6) is expected since we are using only `TV` as a predictor; adding `Radio` would likely improve the model significantly.

---
## Part 6 — K-Means Clustering

### Q6.1 — Data Exploration

In [ ]:
# Load Mall Customer dataset
mall_url = 'https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python'
mall = pd.read_csv(mall_url)


display(mall.head())
mall.info()
display(mall.describe())

In [ ]:
# Scatter plot: Annual Income vs Spending Score
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(mall['Annual Income (k$)'], mall['Spending Score (1-100)'],
           color='steelblue', alpha=0.6, s=40, edgecolors='white')
ax.set_title('Annual Income vs Spending Score')
ax.set_xlabel('Annual Income (k$)')
ax.set_ylabel('Spending Score (1-100)')
plt.tight_layout()
plt.show()

**Visual Groupings:**  
From the scatter plot, I can visually identify approximately **5 natural clusters**:
1. Low Income, Low Spending  
2. Low Income, High Spending  
3. Medium Income, Medium Spending  
4. High Income, Low Spending  
5. High Income, High Spending  

These clusters form distinct blobs with visible separation, making k=5 a reasonable initial guess.

### Q6.2 — Apply K-Means

In [ ]:
from sklearn.cluster import KMeans

# Features
X_mall = mall[['Annual Income (k$)', 'Spending Score (1-100)']].values

# Elbow method to confirm optimal k
inertias = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_mall)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_range, inertias, 'bo-', linewidth=2)
ax.set_title('Elbow Method — Optimal K')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
plt.tight_layout()
plt.show()

In [ ]:
# Fit with optimal k=5
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
mall['Cluster'] = kmeans.fit_predict(X_mall)

# Print cluster centroids
centroids = pd.DataFrame(kmeans.cluster_centers_,
                         columns=['Annual Income (k$)', 'Spending Score (1-100)'])
centroids.index.name = 'Cluster'
print("=== Cluster Centroids ===")
display(centroids.round(2))

# Per-cluster statistics
print("\n=== Per-Cluster Summary ===")
cluster_summary = mall.groupby('Cluster').agg(
    Count=('Cluster', 'count'),
    Mean_Annual_Income=('Annual Income (k$)', 'mean'),
    Mean_Spending_Score=('Spending Score (1-100)', 'mean')
).round(2)
display(cluster_summary)

In [ ]:
# Visualize clusters
cluster_colors = ['red', 'blue', 'green', 'orange', 'purple']
fig, ax = plt.subplots(figsize=(8, 5))

for c in range(optimal_k):
    mask = mall['Cluster'] == c
    ax.scatter(mall.loc[mask, 'Annual Income (k$)'],
               mall.loc[mask, 'Spending Score (1-100)'],
               color=cluster_colors[c], alpha=0.7, s=50, label=f'Cluster {c}')

ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
           color='black', marker='X', s=200, zorder=5, label='Centroids')

ax.set_title('K-Means Clustering — Annual Income vs Spending Score')
ax.set_xlabel('Annual Income (k$)')
ax.set_ylabel('Spending Score (1-100)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Part 7 — Neural Networks on MNIST

### Q7.1 — Data Loading & Preprocessing

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Load MNIST
from tensorflow.keras.datasets import mnist
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_test shape: ", X_test.shape)
print("y_test shape: ", y_test.shape)

# Normalize pixel values to [0, 1]
X_train_norm = X_train / 255.0
X_test_norm  = X_test  / 255.0
print("\nPixel range after normalization:", X_train_norm.min(), "to", X_train_norm.max())

In [ ]:
# Display 5x5 grid of 25 random training images
np.random.seed(42)
random_idx = np.random.choice(len(X_train_norm), 25, replace=False)

fig, axes = plt.subplots(5, 5, figsize=(8, 8))
for ax, idx in zip(axes.flatten(), random_idx):
    ax.imshow(X_train_norm[idx], cmap='gray')
    ax.set_title(str(y_train[idx]), fontsize=10)
    ax.axis('off')

plt.suptitle('25 Random MNIST Training Images', fontsize=13)
plt.tight_layout()
plt.show()

### Q7.2 — Feedforward Neural Network

In [ ]:
# Build Feedforward NN
ffnn = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),       # 28x28 -> 784
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(64,  activation='relu'),
    keras.layers.Dense(10,  activation='softmax')
], name='FeedforwardNN')

ffnn.compile(optimizer='adam',
             loss='sparse_categorical_crossentropy',
             metrics=['accuracy'])

ffnn.summary()

In [ ]:
# Train for 15 epochs
history_ffnn = ffnn.fit(
    X_train_norm, y_train,
    epochs=15,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Plot training & validation accuracy
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_ffnn.history['accuracy'],     label='Train Accuracy')
ax.plot(history_ffnn.history['val_accuracy'], label='Val Accuracy')
ax.set_title('Feedforward NN — Training vs Validation Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
plt.tight_layout()
plt.show()

# Plot training & validation loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_ffnn.history['loss'],     label='Train Loss')
ax.plot(history_ffnn.history['val_loss'], label='Val Loss')
ax.set_title('Feedforward NN — Training vs Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
plt.tight_layout()
plt.show()

### Q7.3 — Evaluation & Error Analysis

In [ ]:
# Test accuracy and loss
test_loss_ffnn, test_acc_ffnn = ffnn.evaluate(X_test_norm, y_test, verbose=0)
print(f"Test Accuracy: {test_acc_ffnn:.4f}")
print(f"Test Loss:     {test_loss_ffnn:.4f}")

# Predictions
y_pred_probs = ffnn.predict(X_test_norm)
y_pred_ffnn  = np.argmax(y_pred_probs, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_ffnn)

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(range(10))
ax.set_yticklabels(range(10))
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix — Feedforward NN on MNIST')

for i in range(10):
    for j in range(10):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8,
                color='white' if cm[i, j] > cm.max() * 0.5 else 'black')

plt.tight_layout()
plt.show()

In [ ]:
# Find the most confused pair (off-diagonal maximum)
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
max_confusion_idx = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
true_label, pred_label = max_confusion_idx
print(f"Most commonly confused pair: True={true_label} predicted as {pred_label} ({cm[true_label, pred_label]} times)")

In [ ]:
# Show 3 example misclassified images for this pair
misclassified_mask = (y_test == true_label) & (y_pred_ffnn == pred_label)
misclassified_idx  = np.where(misclassified_mask)[0][:3]

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, idx in zip(axes, misclassified_idx):
    ax.imshow(X_test_norm[idx], cmap='gray')
    ax.set_title(f'True: {true_label}\nPred: {pred_label}', fontsize=10)
    ax.axis('off')
plt.suptitle(f'Misclassified Examples: {true_label} → {pred_label}', fontsize=12)
plt.tight_layout()
plt.show()

**Most Confused Digits:**  
Typically **4 and 9** (or **3 and 5**) are the most commonly confused pairs. This happens because these digit pairs share similar stroke patterns — for example, both 4 and 9 have a closed loop at the top and a vertical stroke descending downward. A dense network looking at flattened pixel values struggles to distinguish subtle structural differences that humans recognize through shape and spatial reasoning.

### Q7.4 — Convolutional Neural Network

In [ ]:
# Reshape for CNN: (samples, 28, 28, 1)
X_train_cnn = X_train_norm.reshape(-1, 28, 28, 1)
X_test_cnn  = X_test_norm.reshape(-1, 28, 28, 1)

# Build CNN
cnn = keras.Sequential([
    keras.layers.Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Conv2D(64, kernel_size=(3, 3), activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dense(10,  activation='softmax')
], name='CNN')

cnn.compile(optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy'])

cnn.summary()

In [ ]:
# Train for 10 epochs
history_cnn = cnn.fit(
    X_train_cnn, y_train,
    epochs=10,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Final test accuracy
test_loss_cnn, test_acc_cnn = cnn.evaluate(X_test_cnn, y_test, verbose=0)
print(f"CNN Test Accuracy: {test_acc_cnn:.4f}")
print(f"CNN Test Loss:     {test_loss_cnn:.4f}")

In [ ]:
# Comparison table
ffnn_train_acc = history_ffnn.history['accuracy'][-1]
ffnn_val_acc   = history_ffnn.history['val_accuracy'][-1]
cnn_train_acc  = history_cnn.history['accuracy'][-1]
cnn_val_acc    = history_cnn.history['val_accuracy'][-1]

comparison = pd.DataFrame({
    'Model':       ['Feedforward NN', 'CNN'],
    'Parameters':  [ffnn.count_params(), cnn.count_params()],
    'Train Acc':   [round(ffnn_train_acc, 4), round(cnn_train_acc, 4)],
    'Val Acc':     [round(ffnn_val_acc,   4), round(cnn_val_acc,   4)],
    'Test Acc':    [round(test_acc_ffnn,  4), round(test_acc_cnn,  4)],
    'Epochs':      [15, 10]
})
print("=== Model Comparison ===")
display(comparison)

**Why CNNs Outperform Dense Networks on Image Data:**  
1. **Local feature detection:** A Conv layer applies small filters (e.g., 3×3) that detect edges, corners, and textures by looking at *local spatial regions* — something a Dense layer cannot do since it treats each pixel independently.  
2. **Parameter sharing & translation invariance:** The same filter is applied across the entire image, so the network recognizes a feature (e.g., a curve) regardless of where it appears. Dense layers have no such spatial awareness.  
3. **Hierarchical feature learning:** Multiple Conv+Pool layers build increasingly abstract representations — from edges → shapes → digit parts → full digits — which aligns naturally with how image structure is organized.  
4. **Fewer parameters, better generalization:** Despite learning richer features, CNNs often have fewer parameters than large Dense networks for image tasks, reducing overfitting while capturing spatial structure Dense layers miss entirely.